In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import os
from skimage.filters import threshold_otsu
from skimage.segmentation import clear_border
from skimage.morphology import closing, square
from scipy import ndimage
import cv2
import matplotlib.patches as mpatches
from skimage.color import label2rgb
from skimage.measure import label, regionprops,find_contours
from skimage.draw import polygon_perimeter
from tensorflow.keras.callbacks import EarlyStopping
from patchify import patchify

###  Model 1

In [2]:
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(8, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(16, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")


def create_attention_classifier(input_shape, num_patches, num_classes, embedding_dim=128):
    patch_encoder = create_patch_encoder(input_shape, embedding_dim)
    
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    '''
    # Attention mechanism
    attention = layers.Dense(1)(patch_embeddings)
    attention_weights = layers.Softmax(axis=1,name='attention')(attention)
    
    # Weighted sum of patch embeddings
    weighted_embeddings = layers.Multiply()([patch_embeddings, attention_weights])
    context_vector = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(weighted_embeddings)
    '''
    # Classification head
    x = layers.GlobalAveragePooling1D()(patch_embeddings)
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    return patch_encoder,keras.Model(inputs, outputs, name="attention_classifier")

def extract_patches(image, patch_size, stride):
    patches = tf.image.extract_patches(
        images=tf.expand_dims(image, 0),
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, stride, stride, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    patches = tf.reshape(patches, [-1, patch_size, patch_size, image.shape[-1]])
    return patches

def extract_patches_patchify(image, patch_size, stride):
    # Convert image to NumPy array if it's not already
    if isinstance(image, tf.Tensor):
        image = image.numpy()

    # Patchify the image
    patches = patchify(image, (patch_size, patch_size, image.shape[-1]), step=stride)

    # Reshape the patches to match the expected output
    patches = patches.reshape(-1, patch_size, patch_size, image.shape[-1])
    
    if patches.shape[0] % 2 != 0:
        # Extract overlapping patches by reducing the stride slightly
        stride = stride - patch_size // 2 if stride > patch_size // 2 else stride // 2
        patches = patchify(image, (patch_size, patch_size, image.shape[-1]), step=stride)
        patches = patches.reshape(-1, patch_size, patch_size, image.shape[-1])

        # If it's still odd, remove the last patch to make it even
        if patches.shape[0] % 2 != 0:
            patches = patches[:-1]

    return patches
    
    
def preprocess_patches(patches):
    # Resize patches to 224x224 for ResNet50
    patches = tf.image.resize(patches, (224, 224))
    return preprocess_input(patches)

def classify_and_rank_patches(model, image, patch_size, stride, class_names):
    patches = extract_patches(image, patch_size, stride)
    patches = preprocess_patches(patches)
    patches = tf.expand_dims(patches, axis=0)  # Add batch dimension
    
    class_probs, attention_weights = model.predict(patches)
    predicted_class = tf.argmax(class_probs[0]).numpy()
    
    # Rank patches based on attention weights
    ranked_indices = tf.argsort(attention_weights[0, :, 0], direction="DESCENDING").numpy()
    
    return class_names[predicted_class], ranked_indices, attention_weights[0, :, 0], class_probs[0]

# Create and compile the model
input_shape = (224, 224, 3)  # ResNet50 input shape after resizing
num_patches = 4  # 3x3 grid of patches for 512x512 image with 256x256 patches and 128 stride
num_classes = 3  # 3-class classification
class_names = ['No Stroke', 'Ischemic Stroke', 'Chronic Stroke']  # Adjust these names as needed

encoder_model,model = create_attention_classifier(input_shape, num_patches, num_classes)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Specify losses for each output
    #loss_weights={'classification': 1.0, 'attention': 0.0},  # Apply weights
    metrics={'classification': 'accuracy'},  # Metrics for the 'classification' output
    run_eagerly=True
)

early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=5, 
                               restore_best_weights=True)

### Model 2

In [20]:

def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(8, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(16, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")


def create_attention_classifier(input_shape, num_patches, num_classes, embedding_dim=128):
    patch_encoder = create_patch_encoder(input_shape, embedding_dim)
    
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)

    # Attention mechanism
    attention = layers.Dense(1)(patch_embeddings)
    attention_weights = layers.Softmax(axis=1,name='attention')(attention)
    
    # Weighted sum of patch embeddings
    weighted_embeddings = layers.Multiply()([patch_embeddings, attention_weights])
    context_vector = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(weighted_embeddings)

    # Classification head
    x = layers.Dense(512, activation="relu",name='dense1')(context_vector)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    return patch_encoder,keras.Model(inputs, outputs, name="attention_classifier")

def extract_patches(image, patch_size, stride):
    patches = tf.image.extract_patches(
        images=tf.expand_dims(image, 0),
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, stride, stride, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    patches = tf.reshape(patches, [-1, patch_size, patch_size, image.shape[-1]])
    return patches

def extract_patches_patchify(image, patch_size, stride):
    # Convert image to NumPy array if it's not already
    if isinstance(image, tf.Tensor):
        image = image.numpy()

    # Patchify the image
    patches = patchify(image, (patch_size, patch_size, image.shape[-1]), step=stride)

    # Reshape the patches to match the expected output
    patches = patches.reshape(-1, patch_size, patch_size, image.shape[-1])
    
    if patches.shape[0] % 2 != 0:
        # Extract overlapping patches by reducing the stride slightly
        stride = stride - patch_size // 2 if stride > patch_size // 2 else stride // 2
        patches = patchify(image, (patch_size, patch_size, image.shape[-1]), step=stride)
        patches = patches.reshape(-1, patch_size, patch_size, image.shape[-1])

        # If it's still odd, remove the last patch to make it even
        if patches.shape[0] % 2 != 0:
            patches = patches[:-1]

    return patches
    
    
def preprocess_patches(patches):
    # Resize patches to 224x224 for ResNet50
    patches = tf.image.resize(patches, (224, 224))
    return preprocess_input(patches)

def classify_and_rank_patches(model, image, patch_size, stride, class_names):
    patches = extract_patches(image, patch_size, stride)
    patches = preprocess_patches(patches)
    patches = tf.expand_dims(patches, axis=0)  # Add batch dimension
    
    class_probs, attention_weights = model.predict(patches)
    predicted_class = tf.argmax(class_probs[0]).numpy()
    
    # Rank patches based on attention weights
    ranked_indices = tf.argsort(attention_weights[0, :, 0], direction="DESCENDING").numpy()
    
    return class_names[predicted_class], ranked_indices, attention_weights[0, :, 0], class_probs[0]

# Create and compile the model
input_shape = (224, 224, 3)  # ResNet50 input shape after resizing
num_patches = 4  # 3x3 grid of patches for 512x512 image with 256x256 patches and 128 stride
num_classes = 3  # 3-class classification
class_names = ['No Stroke', 'Ischemic Stroke', 'Chronic Stroke']  # Adjust these names as needed

encoder_model,model = create_attention_classifier(input_shape, num_patches, num_classes)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Specify losses for each output
    #loss_weights={'classification': 1.0, 'attention': 0.0},  # Apply weights
    metrics={'classification': 'accuracy'},  # Metrics for the 'classification' output
    run_eagerly=True
)

early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=5, 
                               restore_best_weights=True)

### Model 3

In [61]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.AveragePooling2D(scale)(inputs)
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(x)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    
    # Multi-scale RVFL
    x = multi_scale_rvfl(inputs)
    
    # Adaptive RVFL
    #x = AdaptiveRVFLLayer(128)(x)
    
    # Residual RVFL Block
    x = residual_rvfl_block(x, 448)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(tf.expand_dims(rvfl, 1))
    x = tf.squeeze(attention, 1)
    
    # Uncertainty-aware Classification
    mean_prediction, variance = uncertainty_aware_classification(x, num_classes)
    
    model = keras.Model(inputs=inputs, outputs=[mean_prediction, variance])
    return model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
model = build_rvfl_dr_model(input_shape, num_classes)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_28 (InputLayer)          │ (None, 4, 224, 224, 3)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_8 (TimeDistributed) │ (None, 4, 256)              │         388,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rvfl_layer_27 (RVFLLayer)            │ (None, 4, 256)              │          65,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape_9 (Reshape)                  │ (None, 256, 4)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dynamic_routing_attention_12         │ (None, 256, 4)              │               0 │
│ (DynamicRoutingAttention)            │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_20 (Flatten)                 │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense1 (Dense)                       │ (None, 512)                 │         524,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense2 (Dense)                       │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ classification (Dense)               │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,111,107 (4.24 MB)

 Trainable params: 1,045,571 (3.99 MB)

 Non-trainable params: 65,536 (256.00 KB)

### Model 4

In [3]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    base_model = keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model(inputs)
    outputs = keras.layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = RVFLLayer(2048)(patch_embeddings)
    reshaped = keras.layers.Reshape((2048, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.AveragePooling2D(scale)(inputs)
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(x)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = RVFLLayer(2048)(patch_embeddings)
    reshaped = keras.layers.Reshape((2048, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    '''
    # Multi-scale RVFL
    x = multi_scale_rvfl(inputs)
    
    # Adaptive RVFL
    #x = AdaptiveRVFLLayer(128)(x)
    
    # Residual RVFL Block
    x = residual_rvfl_block(x, 448)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(tf.expand_dims(rvfl, 1))
    x = tf.squeeze(attention, 1)
    
    # Uncertainty-aware Classification
    mean_prediction, variance = uncertainty_aware_classification(x, num_classes)
    '''
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model



def gradual_unfreeze(model, num_layers_to_unfreeze=10):
    # Get the index of the first trainable layer
    trainable_index = next(i for i, layer in enumerate(model.layers) if layer.trainable)
    
    # Get all layers from the trainable index onwards
    trainable_layers = model.layers[trainable_index:]
    
    # Calculate the number of layers to unfreeze in each step
    layers_per_step = max(1, len(trainable_layers) // num_layers_to_unfreeze)
    
    for i in range(0, len(trainable_layers), layers_per_step):
        for layer in trainable_layers[i:i+layers_per_step]:
            layer.trainable = True
        
        yield model

def train_with_gradual_unfreezing(model, train_gen, val_gen, 
                                  num_layers_to_unfreeze=10, epochs_per_stage=1700, 
                                  learning_rate=1e-4, batch_size=32,patience=5, min_delta=0.001,model_save_path='best_model'):
    best_val_loss = float('inf')
    best_model = None
    
    for i, unfrozen_model in enumerate(gradual_unfreeze(model, num_layers_to_unfreeze)):
        print(f"Training stage {i+1}/{num_layers_to_unfreeze}")
        
        # Compile the model
        unfrozen_model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                               loss='categorical_crossentropy',
                               metrics=['accuracy'])

        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
        stage_model_path = f"{model_save_path}_stage_{i+1}.weights.keras"
        model_checkpoint = keras.callbacks.ModelCheckpoint(stage_model_path,monitor='val_loss', save_best_only=True, mode='min',verbose=1)
        
        # Train the model
        history = unfrozen_model.fit(train_gen, epochs=70, steps_per_epoch=316,validation_data=val_gen, validation_steps=40,callbacks=[early_stopping, model_checkpoint])
        
        stage_best_val_loss = min(history.history['val_loss'])
        if stage_best_val_loss < best_val_loss:
            best_val_loss = stage_best_val_loss
            best_model = keras.models.load_model(stage_model_path)
        
        # Optionally, you can adjust the learning rate for the next stage
        learning_rate *= 0.9  # Reduce learning rate by 10% each stage
    
    best_model.save(f"{model_save_path}.weights.keras")
    print(f"Best model saved to {model_save_path}weights.keras")
    
    return best_model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 4, 224, 224, 3)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_1 (TimeDistributed) │ (None, 4, 1280)             │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rvfl_layer (RVFLLayer)               │ (None, 4, 2048)             │       2,623,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 2048, 4)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dynamic_routing_attention            │ (None, 2048, 4)             │               0 │
│ (DynamicRoutingAttention)            │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 8192)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense1 (Dense)                       │ (None, 512)                 │       4,194,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense2 (Dense)                       │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ classification (Dense)               │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9,208,387 (35.13 MB)

 Trainable params: 6,552,835 (25.00 MB)

 Non-trainable params: 2,655,552 (10.13 MB)

### Model 6

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(512)(patch_embeddings)
    reshaped = keras.layers.Reshape((512, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = RVFLLayer(256)(patch_embeddings)
    DRattention = DynamicRoutingAttention()(rvfl)
    CRattention = cross_attention_module(DRattention, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(CRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    '''
    # Multi-scale RVFL
    x = multi_scale_rvfl(inputs)
    
    # Adaptive RVFL
    #x = AdaptiveRVFLLayer(128)(x)
    
    # Residual RVFL Block
    x = residual_rvfl_block(x, 448)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(tf.expand_dims(rvfl, 1))
    x = tf.squeeze(attention, 1)
    
    # Uncertainty-aware Classification
    mean_prediction, variance = uncertainty_aware_classification(x, num_classes)
    '''
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model



def gradual_unfreeze(model, num_layers_to_unfreeze=10):
    # Get the index of the first trainable layer
    trainable_index = next(i for i, layer in enumerate(model.layers) if layer.trainable)
    
    # Get all layers from the trainable index onwards
    trainable_layers = model.layers[trainable_index:]
    
    # Calculate the number of layers to unfreeze in each step
    layers_per_step = max(1, len(trainable_layers) // num_layers_to_unfreeze)
    
    for i in range(0, len(trainable_layers), layers_per_step):
        for layer in trainable_layers[i:i+layers_per_step]:
            layer.trainable = True
        
        yield model

def train_with_gradual_unfreezing(model, train_gen, val_gen, 
                                  num_layers_to_unfreeze=10, epochs_per_stage=1700, 
                                  learning_rate=1e-4, batch_size=32,patience=5, min_delta=0.001,model_save_path='best_model'):
    best_val_loss = float('inf')
    best_model = None
    
    for i, unfrozen_model in enumerate(gradual_unfreeze(model, num_layers_to_unfreeze)):
        print(f"Training stage {i+1}/{num_layers_to_unfreeze}")
        
        # Compile the model
        unfrozen_model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                               loss='categorical_crossentropy',
                               metrics=['accuracy'])

        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
        stage_model_path = f"{model_save_path}_stage_{i+1}.weights.keras"
        model_checkpoint = keras.callbacks.ModelCheckpoint(stage_model_path,monitor='val_loss', save_best_only=True, mode='min',verbose=1)
        
        # Train the model
        history = unfrozen_model.fit(train_gen, epochs=70, steps_per_epoch=316,validation_data=val_gen, validation_steps=40,callbacks=[early_stopping, model_checkpoint])
        
        stage_best_val_loss = min(history.history['val_loss'])
        if stage_best_val_loss < best_val_loss:
            best_val_loss = stage_best_val_loss
            best_model = keras.models.load_model(stage_model_path)
        
        # Optionally, you can adjust the learning rate for the next stage
        learning_rate *= 0.9  # Reduce learning rate by 10% each stage
    
    best_model.save(f"{model_save_path}.weights.keras")
    print(f"Best model saved to {model_save_path}weights.keras")
    
    return best_model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 4, 224, 224, 3)    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_distributed              │ (None, 4, 256)            │         388,416 │ input_layer[0][0]          │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rvfl_layer (RVFLLayer)        │ (None, 4, 256)            │          65,792 │ time_distributed[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dynamic_routing_attention     │ (None, 4, 256)            │               0 │ rvfl_layer[0][0]           │
│ (DynamicRoutingAttention)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 4, 256)            │          65,792 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 4, 256)            │          65,792 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot (Dot)                     │ (None, 4, 4)              │               0 │ dense[0][0], dense_1[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ true_divide (TrueDivide)      │ (None, 4, 4)              │               0 │ dot[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ softmax (Softmax)             │ (None, 4, 4)              │               0 │ true_divide[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 4, 256)            │          65,792 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_1 (Dot)                   │ (None, 4, 256)            │               0 │ softmax[0][0],             │
│                               │                           │                 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 256)               │               0 │ dot_1[0][0]                │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 256)               │               0 │ global_average_pooling1d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 512)               │         131,584 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 512)               │               0 │ dense1[0][0]               │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 915,267 (3.49 MB)

 Trainable params: 849,731 (3.24 MB)

 Non-trainable params: 65,536 (256.00 KB)

### Model 7

In [51]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

'''
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="same")(x)
    x3 = layers.MaxPooling2D()(x)

    x = layers.Conv2D(32, 5, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 5, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 5, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 35, activation="relu", padding="same")(x)
    x5 = layers.MaxPooling2D()(x)

    x = layers.Conv2D(32, 7, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 7, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 7, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 7, activation="relu", padding="same")(x)
    x7 = layers.MaxPooling2D()(x)

    concate = layers.concatenate([x3, x5, x7])
    outputs = layers.GlobalAveragePooling2D()(concate)
    return keras.Model(inputs, outputs, name="patch_encoder")
'''
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = AdaptiveRVFLLayer(512)
    RVFL = rvfl(patch_embeddings)
    DRattention = DynamicRoutingAttention()(RVFL)
    CRattention = cross_attention_module(DRattention, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(CRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    '''
    # Multi-scale RVFL
    x = multi_scale_rvfl(inputs)
    
    # Adaptive RVFL
    #x = AdaptiveRVFLLayer(128)(x)
    
    # Residual RVFL Block
    x = residual_rvfl_block(x, 448)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(tf.expand_dims(rvfl, 1))
    x = tf.squeeze(attention, 1)
    
    # Uncertainty-aware Classification
    mean_prediction, variance = uncertainty_aware_classification(x, num_classes)
    '''
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model



def gradual_unfreeze(model, num_layers_to_unfreeze=10):
    # Get the index of the first trainable layer
    trainable_index = next(i for i, layer in enumerate(model.layers) if layer.trainable)
    
    # Get all layers from the trainable index onwards
    trainable_layers = model.layers[trainable_index:]
    
    # Calculate the number of layers to unfreeze in each step
    layers_per_step = max(1, len(trainable_layers) // num_layers_to_unfreeze)
    
    for i in range(0, len(trainable_layers), layers_per_step):
        for layer in trainable_layers[i:i+layers_per_step]:
            layer.trainable = True
        
        yield model

def train_with_gradual_unfreezing(model, train_gen, val_gen, 
                                  num_layers_to_unfreeze=10, epochs_per_stage=1700, 
                                  learning_rate=1e-4, batch_size=32,patience=5, min_delta=0.001,model_save_path='best_model'):
    best_val_loss = float('inf')
    best_model = None
    
    for i, unfrozen_model in enumerate(gradual_unfreeze(model, num_layers_to_unfreeze)):
        print(f"Training stage {i+1}/{num_layers_to_unfreeze}")
        
        # Compile the model
        unfrozen_model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                               loss='categorical_crossentropy',
                               metrics=['accuracy'])

        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
        stage_model_path = f"{model_save_path}_stage_{i+1}.weights.keras"
        model_checkpoint = keras.callbacks.ModelCheckpoint(stage_model_path,monitor='val_loss', save_best_only=True, mode='min',verbose=1)
        
        # Train the model
        history = unfrozen_model.fit(train_gen, epochs=70, steps_per_epoch=316,validation_data=val_gen, validation_steps=40,callbacks=[early_stopping, model_checkpoint])
        
        stage_best_val_loss = min(history.history['val_loss'])
        if stage_best_val_loss < best_val_loss:
            best_val_loss = stage_best_val_loss
            best_model = keras.models.load_model(stage_model_path)
        
        # Optionally, you can adjust the learning rate for the next stage
        learning_rate *= 0.9  # Reduce learning rate by 10% each stage
    
    best_model.save(f"{model_save_path}.weights.keras")
    print(f"Best model saved to {model_save_path}weights.keras")
    
    return best_model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)   │ (None, 4, 224, 224, 3)    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_distributed_5            │ (None, 4, 256)            │         388,416 │ input_layer_10[0][0]       │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ adaptive_rvfl_layer_4         │ (None, 4, 512)            │         262,656 │ time_distributed_5[0][0]   │
│ (AdaptiveRVFLLayer)           │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dynamic_routing_attention_5   │ (None, 4, 512)            │               0 │ adaptive_rvfl_layer_4[0][… │
│ (DynamicRoutingAttention)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_7 (Dense)               │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_4 (Dot)                   │ (None, 4, 4)              │               0 │ dense_6[0][0],             │
│                               │                           │                 │ dense_7[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ true_divide_2 (TrueDivide)    │ (None, 4, 4)              │               0 │ dot_4[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ softmax_2 (Softmax)           │ (None, 4, 4)              │               0 │ true_divide_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_8 (Dense)               │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_5 (Dot)                   │ (None, 4, 256)            │               0 │ softmax_2[0][0],           │
│                               │                           │                 │ dense_8[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_2    │ (None, 256)               │               0 │ dot_5[0][0]                │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_5 (Flatten)           │ (None, 256)               │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 512)               │         131,584 │ flatten_5[0][0]            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 1,308,739 (4.99 MB)

 Trainable params: 1,177,667 (4.49 MB)

 Non-trainable params: 131,072 (512.00 KB)

### Model 8

In [23]:
import tensorflow as tf
import numpy as np
import cv2


def wld_descriptor_mod(image, num_neighbors=8, num_orientations=8,num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image, num_neighbors)
    segment_min = diff_excitation.min()
    segment_max = diff_excitation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    diff_excitation_hist, _ = np.histogram(diff_excitation, bins=segment_bins, density=True)
    
    # Compute orientation
    orientation = structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0)
    segment_min = orientation.min()
    segment_max = orientation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    orientation_hist, _ = np.histogram(orientation, bins=segment_bins, density=True)
    
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    # Combine differential excitation and orientation into the WLD descriptor histogram
    if num_diff_exc_bins is None:
        num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    num_orientations=8
    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.array(np.ravel(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        segment_min = np.min(np.ravel(wld_2d_hist))
        segment_max = np.max(np.ravel(wld_2d_hist))
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                segment_hist[np.isnan(segment_hist)] = 0  # Replace NaN values with 0
                segment_hist[np.isinf(segment_hist)] = 0
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image, num_neighbors):

    diff_excitation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            neighbors = []
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1]:
                        neighbors.append(image[ny, nx])
            neighbors = np.array(neighbors)
            diff_excitation[y, x] = np.arctan(np.sum(neighbors - image[y, x]) / (image[y, x]+1e-6))
    return diff_excitation

def structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0):

    # Compute gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute structure tensor components
    ix2 = sobel_x ** 2
    iy2 = sobel_y ** 2
    ixy = sobel_x * sobel_y

    # Smooth the structure tensor components
    kernel = cv2.getGaussianKernel(window_size, sigma)
    ix2 = cv2.sepFilter2D(ix2, -1, kernel, kernel)
    iy2 = cv2.sepFilter2D(iy2, -1, kernel, kernel)
    ixy = cv2.sepFilter2D(ixy, -1, kernel, kernel)

    # Compute eigenvalues and orientations
    orientation = np.zeros_like(image, dtype=np.float64)
    num_orientations=8
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            tensor = np.array([[ix2[y, x], ixy[y, x]], [ixy[y, x], iy2[y, x]]])
            eigenvalues, eigenvectors = np.linalg.eig(tensor)
            orientation[y, x] = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    
    if num_orientations is not None:
        orientation = (orientation % (2 * pi)) * num_orientations / (2 * pi)
        return orientation.astype(int)
    else:
        return orientation.astype(int)

def wld_descriptor_multiscale(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, num_scales=4):

    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor_mod(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

# Now, wrap the function using tf.py_function
def tf_wld_descriptor(image_batch):
    return tf.py_function(func=wld_descriptor_batch, inp=[image_batch], Tout=tf.float32)

# Batch processing for the WLD descriptor
def wld_descriptor_batch(image_batch):
    image_batch = image_batch.numpy()  # Convert from Tensor to NumPy
    batch_wld_descriptors = []
    for image in image_batch:
        descriptor = wld_descriptor_mod(image)
        batch_wld_descriptors.append(descriptor)
    return np.array(batch_wld_descriptors, dtype=np.float32)

# MWLD Layer using tf.py_function
class MWLDLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(MWLDLayer, self).__init__()

    def call(self, inputs):
        wld_features = tf_wld_descriptor(inputs)  # Call the WLD descriptor function
        wld_features.set_shape([None, None])  # Adjust the shape based on your output
        return wld_features
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    mwld_features = MWLDLayer()(inputs)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = AdaptiveRVFLLayer(512)
    RVFL = rvfl(patch_embeddings)
    DRattention = DynamicRoutingAttention()(RVFL)
    CRattention = cross_attention_module(DRattention, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(CRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    '''
    # Multi-scale RVFL
    x = multi_scale_rvfl(inputs)
    
    # Adaptive RVFL
    #x = AdaptiveRVFLLayer(128)(x)
    
    # Residual RVFL Block
    x = residual_rvfl_block(x, 448)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(tf.expand_dims(rvfl, 1))
    x = tf.squeeze(attention, 1)
    
    # Uncertainty-aware Classification
    mean_prediction, variance = uncertainty_aware_classification(x, num_classes)
    '''
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.00000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)   │ (None, 4, 224, 224, 3)    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_distributed_3            │ (None, 4, 256)            │         388,416 │ input_layer_14[0][0]       │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ adaptive_rvfl_layer_3         │ (None, 4, 512)            │         262,656 │ time_distributed_3[0][0]   │
│ (AdaptiveRVFLLayer)           │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dynamic_routing_attention_3   │ (None, 4, 512)            │               0 │ adaptive_rvfl_layer_3[0][… │
│ (DynamicRoutingAttention)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_9 (Dense)               │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_10 (Dense)              │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_6 (Dot)                   │ (None, 4, 4)              │               0 │ dense_9[0][0],             │
│                               │                           │                 │ dense_10[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ true_divide_3 (TrueDivide)    │ (None, 4, 4)              │               0 │ dot_6[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ softmax_3 (Softmax)           │ (None, 4, 4)              │               0 │ true_divide_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_11 (Dense)              │ (None, 4, 256)            │         131,328 │ dynamic_routing_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_7 (Dot)                   │ (None, 4, 256)            │               0 │ softmax_3[0][0],           │
│                               │                           │                 │ dense_11[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_3    │ (None, 256)               │               0 │ dot_7[0][0]                │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_3 (Flatten)           │ (None, 256)               │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 512)               │         131,584 │ flatten_3[0][0]            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 1,308,739 (4.99 MB)

 Trainable params: 1,177,667 (4.49 MB)

 Non-trainable params: 131,072 (512.00 KB)

### Model 10

In [37]:
import tensorflow as tf
import numpy as np
import cv2


def wld_descriptor_mod(image, num_neighbors=8, num_orientations=8,num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image, num_neighbors)
    segment_min = diff_excitation.min()
    segment_max = diff_excitation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    diff_excitation_hist, _ = np.histogram(diff_excitation, bins=segment_bins, density=True)
    
    # Compute orientation
    orientation = structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0)
    segment_min = orientation.min()
    segment_max = orientation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    orientation_hist, _ = np.histogram(orientation, bins=segment_bins, density=True)
    
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    # Combine differential excitation and orientation into the WLD descriptor histogram
    if num_diff_exc_bins is None:
        num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    num_orientations=8
    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.array(np.ravel(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        segment_min = np.min(np.ravel(wld_2d_hist))
        segment_max = np.max(np.ravel(wld_2d_hist))
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                segment_hist[np.isnan(segment_hist)] = 0  # Replace NaN values with 0
                segment_hist[np.isinf(segment_hist)] = 0
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image, num_neighbors):

    diff_excitation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            neighbors = []
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1]:
                        neighbors.append(image[ny, nx])
            neighbors = np.array(neighbors)
            diff_excitation[y, x] = np.arctan(np.sum(neighbors - image[y, x]) / (image[y, x]+1e-6))
    return diff_excitation

def structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0):

    # Compute gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute structure tensor components
    ix2 = sobel_x ** 2
    iy2 = sobel_y ** 2
    ixy = sobel_x * sobel_y

    # Smooth the structure tensor components
    kernel = cv2.getGaussianKernel(window_size, sigma)
    ix2 = cv2.sepFilter2D(ix2, -1, kernel, kernel)
    iy2 = cv2.sepFilter2D(iy2, -1, kernel, kernel)
    ixy = cv2.sepFilter2D(ixy, -1, kernel, kernel)

    # Compute eigenvalues and orientations
    orientation = np.zeros_like(image, dtype=np.float64)
    num_orientations=8
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            tensor = np.array([[ix2[y, x], ixy[y, x]], [ixy[y, x], iy2[y, x]]])
            eigenvalues, eigenvectors = np.linalg.eig(tensor)
            orientation[y, x] = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    
    if num_orientations is not None:
        orientation = (orientation % (2 * pi)) * num_orientations / (2 * pi)
        return orientation.astype(int)
    else:
        return orientation.astype(int)

def wld_descriptor_multiscale(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, num_scales=4):

    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor_mod(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

# Now, wrap the function using tf.py_function
def tf_wld_descriptor(image_batch):
    return tf.py_function(func=wld_descriptor_batch, inp=[image_batch], Tout=tf.float32)

# Batch processing for the WLD descriptor
def wld_descriptor_batch(image_batch):
    image_batch = image_batch.numpy()  # Convert from Tensor to NumPy
    batch_wld_descriptors = []
    for image in image_batch:
        descriptor = wld_descriptor_mod(image)
        batch_wld_descriptors.append(descriptor)
    return np.array(batch_wld_descriptors, dtype=np.float32)

# MWLD Layer using tf.py_function
class MWLDLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(MWLDLayer, self).__init__()

    def call(self, inputs):
        wld_features = tf_wld_descriptor(inputs)  # Call the WLD descriptor function
        wld_features.set_shape([None, None])  # Adjust the shape based on your output
        return wld_features
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    mwld_features = MWLDLayer()(inputs)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = AdaptiveRVFLLayer(256)
    RVFL = rvfl(patch_embeddings)
    #DRattention = DynamicRoutingAttention()(RVFL)
    CRattention = cross_attention_module(RVFL, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(CRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.00000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_18 (InputLayer)   │ (None, 4, 224, 224, 3)    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_distributed_5            │ (None, 4, 256)            │         388,416 │ input_layer_18[0][0]       │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ adaptive_rvfl_layer_5         │ (None, 4, 256)            │         131,328 │ time_distributed_5[0][0]   │
│ (AdaptiveRVFLLayer)           │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_15 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_5[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_16 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_5[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_10 (Dot)                  │ (None, 4, 4)              │               0 │ dense_15[0][0],            │
│                               │                           │                 │ dense_16[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ true_divide_5 (TrueDivide)    │ (None, 4, 4)              │               0 │ dot_10[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ softmax_5 (Softmax)           │ (None, 4, 4)              │               0 │ true_divide_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_17 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_5[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_11 (Dot)                  │ (None, 4, 256)            │               0 │ softmax_5[0][0],           │
│                               │                           │                 │ dense_17[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_5    │ (None, 256)               │               0 │ dot_11[0][0]               │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_5 (Flatten)           │ (None, 256)               │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 512)               │         131,584 │ flatten_5[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_10 (Dropout)          │ (None, 512)               │               0 │ dense1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense2 (Dense)                │ (None, 256)               │         131,32

 Total params: 980,803 (3.74 MB)

 Trainable params: 915,267 (3.49 MB)

 Non-trainable params: 65,536 (256.00 KB)

### Model  6 (2) without preprocessing

In [57]:
import tensorflow as tf
import numpy as np
import cv2


def wld_descriptor_mod(image, num_neighbors=8, num_orientations=8,num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image, num_neighbors)
    segment_min = diff_excitation.min()
    segment_max = diff_excitation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    diff_excitation_hist, _ = np.histogram(diff_excitation, bins=segment_bins, density=True)
    
    # Compute orientation
    orientation = structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0)
    segment_min = orientation.min()
    segment_max = orientation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    orientation_hist, _ = np.histogram(orientation, bins=segment_bins, density=True)
    
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    # Combine differential excitation and orientation into the WLD descriptor histogram
    if num_diff_exc_bins is None:
        num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    num_orientations=8
    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.array(np.ravel(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        segment_min = np.min(np.ravel(wld_2d_hist))
        segment_max = np.max(np.ravel(wld_2d_hist))
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                segment_hist[np.isnan(segment_hist)] = 0  # Replace NaN values with 0
                segment_hist[np.isinf(segment_hist)] = 0
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image, num_neighbors):

    diff_excitation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            neighbors = []
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1]:
                        neighbors.append(image[ny, nx])
            neighbors = np.array(neighbors)
            diff_excitation[y, x] = np.arctan(np.sum(neighbors - image[y, x]) / (image[y, x]+1e-6))
    return diff_excitation

def structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0):

    # Compute gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute structure tensor components
    ix2 = sobel_x ** 2
    iy2 = sobel_y ** 2
    ixy = sobel_x * sobel_y

    # Smooth the structure tensor components
    kernel = cv2.getGaussianKernel(window_size, sigma)
    ix2 = cv2.sepFilter2D(ix2, -1, kernel, kernel)
    iy2 = cv2.sepFilter2D(iy2, -1, kernel, kernel)
    ixy = cv2.sepFilter2D(ixy, -1, kernel, kernel)

    # Compute eigenvalues and orientations
    orientation = np.zeros_like(image, dtype=np.float64)
    num_orientations=8
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            tensor = np.array([[ix2[y, x], ixy[y, x]], [ixy[y, x], iy2[y, x]]])
            eigenvalues, eigenvectors = np.linalg.eig(tensor)
            orientation[y, x] = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    
    if num_orientations is not None:
        orientation = (orientation % (2 * pi)) * num_orientations / (2 * pi)
        return orientation.astype(int)
    else:
        return orientation.astype(int)

def wld_descriptor_multiscale(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, num_scales=4):

    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor_mod(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

# Now, wrap the function using tf.py_function
def tf_wld_descriptor(image_batch):
    return tf.py_function(func=wld_descriptor_batch, inp=[image_batch], Tout=tf.float32)

# Batch processing for the WLD descriptor
def wld_descriptor_batch(image_batch):
    image_batch = image_batch.numpy()  # Convert from Tensor to NumPy
    batch_wld_descriptors = []
    for image in image_batch:
        descriptor = wld_descriptor_mod(image)
        batch_wld_descriptors.append(descriptor)
    return np.array(batch_wld_descriptors, dtype=np.float32)

# MWLD Layer using tf.py_function
class MWLDLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(MWLDLayer, self).__init__()

    def call(self, inputs):
        wld_features = tf_wld_descriptor(inputs)  # Call the WLD descriptor function
        wld_features.set_shape([None, None])  # Adjust the shape based on your output
        return wld_features
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    mwld_features = MWLDLayer()(inputs)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = AdaptiveRVFLLayer(256)
    RVFL = rvfl(patch_embeddings)
    #DRattention = DynamicRoutingAttention()(RVFL)
    CRattention = cross_attention_module(RVFL, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(CRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.00000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)   │ (None, 4, 224, 224, 3)    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ time_distributed_7            │ (None, 4, 256)            │         388,416 │ input_layer_14[0][0]       │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ adaptive_rvfl_layer_6         │ (None, 4, 256)            │         131,328 │ time_distributed_7[0][0]   │
│ (AdaptiveRVFLLayer)           │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_12 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_6[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_13 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_6[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_8 (Dot)                   │ (None, 4, 4)              │               0 │ dense_12[0][0],            │
│                               │                           │                 │ dense_13[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ true_divide_4 (TrueDivide)    │ (None, 4, 4)              │               0 │ dot_8[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ softmax_4 (Softmax)           │ (None, 4, 4)              │               0 │ true_divide_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_14 (Dense)              │ (None, 4, 256)            │          65,792 │ adaptive_rvfl_layer_6[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot_9 (Dot)                   │ (None, 4, 256)            │               0 │ softmax_4[0][0],           │
│                               │                           │                 │ dense_14[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_4    │ (None, 256)               │               0 │ dot_9[0][0]                │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_7 (Flatten)           │ (None, 256)               │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 512)               │         131,584 │ flatten_7[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_12 (Dropout)          │ (None, 512)               │               0 │ dense1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense2 (Dense)                │ (None, 256)               │         131,32

 Total params: 980,803 (3.74 MB)

 Trainable params: 915,267 (3.49 MB)

 Non-trainable params: 65,536 (256.00 KB)

### Model 6 (3)

In [34]:
import tensorflow as tf
import numpy as np
import cv2


def wld_descriptor_mod(image, num_neighbors=8, num_orientations=8,num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image, num_neighbors)
    segment_min = diff_excitation.min()
    segment_max = diff_excitation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    diff_excitation_hist, _ = np.histogram(diff_excitation, bins=segment_bins, density=True)
    
    # Compute orientation
    orientation = structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0)
    segment_min = orientation.min()
    segment_max = orientation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    orientation_hist, _ = np.histogram(orientation, bins=segment_bins, density=True)
    
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    # Combine differential excitation and orientation into the WLD descriptor histogram
    if num_diff_exc_bins is None:
        num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    num_orientations=8
    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.array(np.ravel(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        segment_min = np.min(np.ravel(wld_2d_hist))
        segment_max = np.max(np.ravel(wld_2d_hist))
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                segment_hist[np.isnan(segment_hist)] = 0  # Replace NaN values with 0
                segment_hist[np.isinf(segment_hist)] = 0
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image, num_neighbors):

    diff_excitation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            neighbors = []
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1]:
                        neighbors.append(image[ny, nx])
            neighbors = np.array(neighbors)
            diff_excitation[y, x] = np.arctan(np.sum(neighbors - image[y, x]) / (image[y, x]+1e-6))
    return diff_excitation

def structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0):

    # Compute gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute structure tensor components
    ix2 = sobel_x ** 2
    iy2 = sobel_y ** 2
    ixy = sobel_x * sobel_y

    # Smooth the structure tensor components
    kernel = cv2.getGaussianKernel(window_size, sigma)
    ix2 = cv2.sepFilter2D(ix2, -1, kernel, kernel)
    iy2 = cv2.sepFilter2D(iy2, -1, kernel, kernel)
    ixy = cv2.sepFilter2D(ixy, -1, kernel, kernel)

    # Compute eigenvalues and orientations
    orientation = np.zeros_like(image, dtype=np.float64)
    num_orientations=8
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            tensor = np.array([[ix2[y, x], ixy[y, x]], [ixy[y, x], iy2[y, x]]])
            eigenvalues, eigenvectors = np.linalg.eig(tensor)
            orientation[y, x] = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    
    if num_orientations is not None:
        orientation = (orientation % (2 * pi)) * num_orientations / (2 * pi)
        return orientation.astype(int)
    else:
        return orientation.astype(int)

def wld_descriptor_multiscale(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, num_scales=4):

    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor_mod(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

# Now, wrap the function using tf.py_function
def tf_wld_descriptor(image_batch):
    return tf.py_function(func=wld_descriptor_batch, inp=[image_batch], Tout=tf.float32)

# Batch processing for the WLD descriptor
def wld_descriptor_batch(image_batch):
    image_batch = image_batch.numpy()  # Convert from Tensor to NumPy
    batch_wld_descriptors = []
    for image in image_batch:
        descriptor = wld_descriptor_mod(image)
        batch_wld_descriptors.append(descriptor)
    return np.array(batch_wld_descriptors, dtype=np.float32)

# MWLD Layer using tf.py_function
class MWLDLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(MWLDLayer, self).__init__()

    def call(self, inputs):
        wld_features = tf_wld_descriptor(inputs)  # Call the WLD descriptor function
        wld_features.set_shape([None, None])  # Adjust the shape based on your output
        return wld_features
def create_patch_encoder(input_shape, embedding_dim):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation="relu", padding="valid")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="valid")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(embedding_dim, 3, activation="relu", padding="valid")(x)
    outputs = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, outputs, name="patch_encoder")
                       
class RVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(RVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

class DynamicRoutingAttention(keras.layers.Layer):
    def __init__(self, num_iterations=3):
        super(DynamicRoutingAttention, self).__init__()
        self.num_iterations = num_iterations

    def build(self, input_shape):
        self.num_capsules = input_shape[1]
        self.capsule_dim = input_shape[2]

    def call(self, inputs):
        b = tf.zeros((tf.shape(inputs)[0], self.num_capsules, self.num_capsules))
        
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=2)
            s = tf.matmul(c, inputs)
            v = self.squash(s)
            b += tf.matmul(inputs, v, transpose_b=True)
        
        return v

    def squash(self, s):
        s_squared_norm = tf.reduce_sum(tf.square(s), axis=-1, keepdims=True)
        return (s_squared_norm / (1 + s_squared_norm)) * (s / tf.sqrt(s_squared_norm + 1e-8))

def cross_attention_module(patch_embeddings,num_patches, embedding_dim):
    """
    Compute cross-attention for a set of image patches.

    Args:
    patch_embeddings: A tensor of shape (batch_size, num_patches, embedding_dim).

    Returns:
    weighted_sum: A tensor of shape (batch_size, embedding_dim).
    """
    # Compute query, key, and value
    queries = layers.Dense(embedding_dim)(patch_embeddings)  # (batch_size, num_patches, embedding_dim)
    keys = layers.Dense(embedding_dim)(patch_embeddings)      # (batch_size, num_patches, embedding_dim)
    values = layers.Dense(embedding_dim)(patch_embeddings)    # (batch_size, num_patches, embedding_dim)

    # Scaled dot-product attention
    attention_scores = layers.Dot(axes=(-1, -1))([queries, keys])  # Use Dot layer for matmul
    attention_scores = attention_scores / tf.sqrt(tf.cast(embedding_dim, tf.float32))

    # Softmax to get attention weights
    attention_weights = layers.Softmax(axis=-1)(attention_scores)

    # Weighted sum of values
    weighted_sum = layers.Dot(axes=(1, 1))([attention_weights, values])  # (batch_size, num_patches, embedding_dim)
    
    # Aggregate over patches (mean, sum, or other method)
    output = layers.GlobalAveragePooling1D()(weighted_sum)

    return output

def build_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    patch_encoder.summary()

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    
    rvfl = RVFLLayer(256)(patch_embeddings)
    reshaped = keras.layers.Reshape((256, 4))(rvfl)
    
    # Dynamic Routing Attention
    attention = DynamicRoutingAttention()(reshaped)
    x = keras.layers.Flatten()(attention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Novel feature: Multi-scale RVFL
def multi_scale_rvfl(inputs, scales=[1, 2, 4]):
    multi_scale_features = []
    for scale in scales:
        x = keras.layers.Conv2D(32 * scale, 3, activation='relu')(inputs)
        x = keras.layers.Flatten()(x)
        x = RVFLLayer(64 * scale)(x)
        multi_scale_features.append(x)
    return keras.layers.Concatenate()(multi_scale_features)

# Novel feature: Adaptive Random Projection
class AdaptiveRVFLLayer(keras.layers.Layer):
    def __init__(self, units, activation='relu'):
        super(AdaptiveRVFLLayer, self).__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=False
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.adaptive_factor = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        adapted_w = self.w * self.adaptive_factor
        return self.activation(tf.matmul(inputs, adapted_w) + self.b)

# Novel feature: Residual RVFL Block
def residual_rvfl_block(inputs, units):
    x = RVFLLayer(units)(inputs)
    x = RVFLLayer(units)(x)
    return keras.layers.Add()([inputs, x])

# Novel feature: Uncertainty-aware Classification
def uncertainty_aware_classification(x, num_classes, num_samples=10):
    def sample_model():
        return keras.layers.Dense(num_classes, activation='softmax')(x)
    
    samples = [sample_model() for _ in range(num_samples)]
    mean_prediction = tf.reduce_mean(samples, axis=0)
    variance = tf.reduce_mean(tf.square(samples - mean_prediction), axis=0)
    return mean_prediction, variance

# Example of how to use these novel features in a model
def build_advanced_rvfl_dr_model(input_shape, num_classes):
    inputs = keras.Input(shape=(num_patches,) + input_shape)
    patch_encoder = create_patch_encoder(input_shape, embedding_dim=256)
    #mwld_features = MWLDLayer()(inputs)

    patch_embeddings = layers.TimeDistributed(patch_encoder)(inputs)
    rvfl = AdaptiveRVFLLayer(256)
    RVFL = rvfl(patch_embeddings)
    DRattention = DynamicRoutingAttention()(RVFL)
    #CRattention = cross_attention_module(RVFL, num_patches=4, embedding_dim=256)
    #x = layers.concatenate((DRattention,CRattention),axis=-1)


    #rvfl takes to higher dimensions order, dr tries to find patterns and cross get more of it
    
    # Dynamic Routing Attention
    
    x = keras.layers.Flatten()(DRattention)
    
    x = layers.Dense(512, activation="relu",name='dense1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",name='dense2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax",name='classification')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Usage example
input_shape = (224, 224, 3)  # Example input shape for CT images
num_classes = 3
num_patches = 4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,patience=3, min_lr=0.00000000000000001)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
model = build_advanced_rvfl_dr_model(input_shape, num_classes)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)           │ (None, 4, 224, 224, 3)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_3 (TimeDistributed) │ (None, 4, 256)              │         388,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ adaptive_rvfl_layer_2                │ (None, 4, 256)              │         131,328 │
│ (AdaptiveRVFLLayer)                  │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dynamic_routing_attention_3          │ (None, 4, 256)              │               0 │
│ (DynamicRoutingAttention)            │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense1 (Dense)                       │ (None, 512)                 │         524,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense2 (Dense)                       │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ classification (Dense)               │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,176,643 (4.49 MB)

 Trainable params: 1,111,107 (4.24 MB)

 Non-trainable params: 65,536 (256.00 KB)

In [58]:
def preprocessing(im1):
    
    im1 = cv2.convertScaleAbs(im1)
    if len(im1.shape) == 2:
        # Grayscale image
        im1[im1 == 255] = 0
        im2 = cv2.medianBlur(im1, 51)
        th, img2 = cv2.threshold(im2, 10, 255, cv2.THRESH_BINARY)
        img2[img2 == 255] = 1
        image = np.multiply(img2, im1)
    else:
        # RGB image
        im1 = cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY)
        im1[im1 == 255] = 0
        im2 = cv2.medianBlur(im1, 51)
        th, img2 = cv2.threshold(im2, 10, 255, cv2.THRESH_BINARY)
        img2[img2 == 255] = 1
        image=np.multiply(img2,im1)

    
    thresh = threshold_otsu(image)
    bw = closing(image > thresh, square(3))
    cleared = clear_border(bw)
    label_image = label(cleared)
    image_label_overlay = label2rgb(label_image, image=image, bg_label=0)
    contours = find_contours(image, 0.8)
    bounding_boxes = []
    
    for contour in contours:
        Xmin = np.min(contour[:,0])
        Xmax = np.max(contour[:,0])
        Ymin = np.min(contour[:,1])
        Ymax = np.max(contour[:,1])

        bounding_boxes.append([Xmin, Xmax, Ymin, Ymax])

    with_boxes  = np.copy(image)

    for box in bounding_boxes:
        #[Xmin, Xmax, Ymin, Ymax]
        r = [box[0],box[1],box[1],box[0], box[0]]
        c = [box[3],box[3],box[2],box[2], box[3]]
        rr, cc = polygon_perimeter(r, c, with_boxes.shape)
        with_boxes[rr, cc] = 5 #set color white
        
    #fig, ax = plt.subplots(figsize=(10, 6))
    poly=[]
    
    for region in regionprops(label_image):
        # take regions with large enough areas
        if region.area >= 10000:
                # draw rectangle around segmented coins
            minr, minc, maxr, maxc = region.bbox
            rect = mpatches.Rectangle((minc, minr), maxc - minc, maxr - minr,
                                          fill=False, edgecolor='red', linewidth=2)
            poly.append(region.bbox)
            #ax.add_patch(rect)

    try:
        a,b,c,d=poly[-1]
        return cv2.resize(image[a:c,b:d], (512,512), interpolation= cv2.INTER_LINEAR)
    except IndexError:
        return cv2.resize(image, (512,512), interpolation= cv2.INTER_LINEAR)

In [59]:
dir_train='C:/Users/MAHE/Documents/Mahesh/data/'

def resize_image(image, target_size=(512, 512)):
    resized_image = cv2.resize(image, target_size)
    return resized_image

def combine(img):
    return cv2.merge((img,img,img))

def get_label(dir):
    if 'AS' in dir:
        return 1
    elif 'CS' in dir:
        return 2
    else:
        return 0

def preprocess_patches(patches):
    # Resize patches to 224x224 for ResNet50
    patches = tf.image.resize(patches, (224, 224))
    return preprocess_input(patches)

In [60]:
import os
import numpy as np
import tensorflow as tf
import cv2
from patchify import patchify
import albumentations as A
from sklearn.model_selection import train_test_split

def create_data_generators(dir_path, batch_size=16, image_size=(512, 512), patch_size=224, validation_split=0.1, test_split=0.1):
    image_paths = []
    labels = []
    
    # Collect all image paths and labels
    for root, _, files in os.walk(dir_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
                image_paths.append(os.path.join(root, file))
                labels.append(get_label(file))
            
    
    # First split: Training + Validation vs Test
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        image_paths, labels, test_size=test_split, stratify=labels, random_state=42
    )
    
    # Second split: Training vs Validation
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_val_paths, train_val_labels, test_size=validation_split, stratify=train_val_labels, random_state=42
    )

    augmentation = A.Compose([
        A.HorizontalFlip(p=0.75),
        A.VerticalFlip(p=0.75),
        A.augmentations.geometric.rotate.Rotate(limit=15, p=0.5),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(224, 224), always_apply=False, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    ])

    def generator(paths, labels, is_training):
        total_images = len(paths)
        
        while True:
            # Shuffle the dataset for each epoch
            indices = np.arange(total_images)
            if is_training:
                np.random.shuffle(indices)
            
            for start in range(0, total_images, batch_size):
                end = min(start + batch_size, total_images)
                batch_indices = indices[start:end]
                
                batch_images = []
                batch_labels = []
                
                for i in batch_indices:
                    try:
                        img = cv2.imread(paths[i], cv2.IMREAD_COLOR)
                        img = cv2.resize(img, image_size)  # Ensure consistent size
                        if is_training:
                            augmented = augmentation(image=img)
                            img = augmented['image']
                        img = preprocessing(img)  # Assuming this function exists
                        img = combine(img)
                        
                        # Extract patches
                        patches = patchify(img, (patch_size, patch_size, 3), step=patch_size)
                        patches = patches.reshape(-1, patch_size, patch_size, 3)
                        
                        # Preprocess patches
                        patches = preprocess_patches(patches)
                        
                        batch_images.append(patches)
                        batch_labels.append(labels[i])
                    except Exception as e:
                        print(f"Error processing image {paths[i]}: {str(e)}")
                        continue
                
                if not batch_images:
                    continue  # Skip this batch if all images failed to process
                
                X = np.array(batch_images)
                y = tf.keras.utils.to_categorical(batch_labels, num_classes=3)
                
                # Create a dummy tensor for attention weights
                attention_weights = np.zeros((len(batch_labels), X.shape[1], 1))
                
                yield X, (y, attention_weights)

    train_gen = generator(train_paths, train_labels, is_training=True)
    val_gen = generator(val_paths, val_labels, is_training=False)
    test_gen = generator(test_paths, test_labels, is_training=False)

    return train_gen, val_gen, test_gen

# Usage
train_gen, val_gen, test_gen = create_data_generators(dir_train, validation_split=0.1, test_split=0.1)


In [61]:
history = model.fit(train_gen, epochs=70, steps_per_epoch=316,validation_data=val_gen, validation_steps=40,callbacks=[early_stopping,reduce_lr]) #callbacks=[early_stopping]

Epoch 1/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 1026s 3s/step - accuracy: 0.7139 - loss: 1.0285 - val_accuracy: 0.7417 - val_loss: 0.7041 - learning_rate: 0.0010
Epoch 2/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 988s 3s/step - accuracy: 0.7362 - loss: 0.7518 - val_accuracy: 0.7274 - val_loss: 0.7140 - learning_rate: 0.0010
Epoch 3/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 987s 3s/step - accuracy: 0.7404 - loss: 0.7328 - val_accuracy: 0.7444 - val_loss: 0.6800 - learning_rate: 0.0010
Epoch 4/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 984s 3s/step - accuracy: 0.7294 - loss: 0.7419 - val_accuracy: 0.7258 - val_loss: 0.7094 - learning_rate: 0.0010
Epoch 5/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 976s 3s/step - accuracy: 0.7397 - loss: 0.7148 - val_accuracy: 0.7910 - val_loss: 0.6039 - learning_rate: 0.0010
Epoch 6/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 966s 3s/step - accuracy: 0.7491 - loss: 0.7049 - val_accuracy: 0.7274 - val_loss: 0.7259 - learning_rate: 0.0010
Epoch 7/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 999s 3s/step - accuracy: 0.7543 - loss: 0

In [62]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf

def evaluate_model_on_test_set(model, test_gen, steps, num_classes=3):
    all_labels = []
    all_predictions = []

    # Loop through the test set
    for _ in range(steps):
        X_batch, (y_batch, _) = next(test_gen)
        
        # Predict the output for the batch
        predictions = model.predict(X_batch, verbose=0)
        
        # Get the predicted class (argmax)
        pred_classes = np.argmax(predictions, axis=1)
        
        # Get the true class (argmax)
        true_classes = np.argmax(y_batch, axis=1)
        
        # Append to the lists
        all_labels.extend(true_classes)
        all_predictions.extend(pred_classes)
    
    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    # Calculate accuracy, precision, recall, and F1 score
    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average='weighted')
    recall = recall_score(all_labels, all_predictions, average='weighted')
    f1 = f1_score(all_labels, all_predictions, average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return accuracy, precision, recall, f1

# Assuming you have a test generator and your trained model
# Example usage:
# steps is the number of batches to evaluate. Adjust it according to the size of your test set.
test_steps = 40  # Calculate as len(test_dataset) // batch_size
accuracy, precision, recall, f1 = evaluate_model_on_test_set(model, test_gen, steps=test_steps)


Accuracy: 0.7635
Precision: 0.7047
Recall: 0.7635
F1 Score: 0.6879


In [72]:
model.save_weights('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(3)/with preprocessing/model63.weights.h5')

AttributeError: 'CatBoostClassifier' object has no attribute 'save_weights'

In [64]:
from keras import backend as K
import tqdm
from keras.models import Model

# with a Sequential model
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer('dense1').output)
image_size=(512,512)
patch_size=224
X=np.zeros((4698,513))
index=0
for file in tqdm.tqdm(os.listdir(dir_train)):
    img = cv2.imread(os.path.join(dir_train,file))
    img = cv2.resize(img, image_size)  # Ensure consistent size
    #img = preprocessing(img)  # Assuming this function exists
    img = combine(img)
    patches = patchify(img, (patch_size, patch_size, 3), step=patch_size)
    patches = patches.reshape(-1, patch_size, patch_size, 3)
    X[index,0:512]=intermediate_layer_model.predict([np.expand_dims(patches,0)])[0]
    X[index,-1]=get_label(file)
    index+=1

np.savetxt("C:/Users/MAHE/Documents/Mahesh/model 6/model 6(3)/512.csv", X, delimiter=",")

# with a Sequential model
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer('dense2').output)
image_size=(512,512)
patch_size=224
X=np.zeros((4698,257))
index=0
for file in tqdm.tqdm(os.listdir(dir_train)):
    img = cv2.imread(os.path.join(dir_train,file))
    img = cv2.resize(img, image_size)  # Ensure consistent size
    #img = preprocessing(img)  # Assuming this function exists
    img = combine(img)
    patches = patchify(img, (patch_size, patch_size, 3), step=patch_size)
    patches = patches.reshape(-1, patch_size, patch_size, 3)
    X[index,0:256]=intermediate_layer_model.predict([np.expand_dims(patches,0)])[0]
    X[index,-1]=get_label(file)
    index+=1

np.savetxt("C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/256.csv", X, delimiter=",")

  0%|                                                                                         | 0/4698 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step


  0%|                                                                                 | 1/4698 [00:00<19:31,  4.01it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


  0%|                                                                                 | 3/4698 [00:00<09:35,  8.16it/s]

...

100%|█████████████████████████████████████████████████████████████████████████████▉| 4695/4698 [06:11<00:00, 12.79it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


100%|█████████████████████████████████████████████████████████████████████████████▉| 4697/4698 [06:11<00:00, 12.42it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


100%|██████████████████████████████████████████████████████████████████████████████| 4698/4698 [06:11<00:00, 12.64it/s]


In [65]:
df=pd.read_csv('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/512.csv')
X=df.iloc[:,0:512].values
y=df.iloc[:,-1].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

clf1 = XGBClassifier(random_state=1)
clf2 = RandomForestClassifier(n_estimators=100, random_state=1)
clf3 = clf3 = lgb.LGBMClassifier(boosting_type='gbdt',
                                 objective='multiclass',
                                 num_class=3,
                                 num_leaves=31,
                                 learning_rate=0.1,
                                 n_estimators=100)

eclf1 = VotingClassifier(estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)], voting='hard')
eclf1 = eclf1.fit(X_train, y_train)
y_pred=eclf1.predict(X_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005439 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 45459
[LightGBM] [Info] Number of data points in the train set: 3757, number of used features: 279
[LightGBM] [Info] Start training from score -0.308390
[LightGBM] [Info] Start training from score -1.846182
[LightGBM] [Info] Start training from score -2.229961
Accuracy (0.9223404255319149,) Precision (0.9264244195273282,) Recall (0.9223404255319149,) F1 (0.9176650725063437,)


In [66]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, verbose=0)
df=pd.read_csv('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/512.csv')
X=df.iloc[:,0:512].values
y=df.iloc[:,-1].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model.fit(X_train, y_train)
y_pred=model.predict(X_test)

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)


Accuracy (0.9319148936170213,) Precision (0.9343230609407267,) Recall (0.9319148936170213,) F1 (0.9286894919878146,)


In [67]:
df=pd.read_csv('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/256.csv')
X=df.iloc[:,0:256].values
y=df.iloc[:,-1].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

clf1 = XGBClassifier(random_state=1)
clf2 = RandomForestClassifier(n_estimators=100, random_state=1)
clf3 = lgb.LGBMClassifier(boosting_type='gbdt',
                                 objective='multiclass',
                                 num_class=3,
                                 num_leaves=31,
                                 learning_rate=0.1,
                                 n_estimators=100)

eclf1 = VotingClassifier(estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)], voting='hard')
eclf1 = eclf1.fit(X_train, y_train)
y_pred=eclf1.predict(X_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007041 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21388
[LightGBM] [Info] Number of data points in the train set: 3757, number of used features: 138
[LightGBM] [Info] Start training from score -0.308390
[LightGBM] [Info] Start training from score -1.846182
[LightGBM] [Info] Start training from score -2.229961
Accuracy (0.8659574468085106,) Precision (0.8678687890349427,) Recall (0.8659574468085106,) F1 (0.8537270929554078,)


In [70]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, verbose=0)
model.fit(X_train, y_train)
y_pred=model.predict(X_test)

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)


Accuracy (0.8712765957446809,) Precision (0.8738353592131717,) Recall (0.8712765957446809,) F1 (0.8602244835098739,)


In [71]:
df=pd.read_csv('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/512.csv')
X=df.iloc[:,0:512].values
y=df.iloc[:,-1].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train, y_train)
y_pred=neigh.predict(X_test)

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)


Accuracy (0.7489361702127659,) Precision (0.7233640474954253,) Recall (0.7489361702127659,) F1 (0.7288832463837973,)


In [69]:
df=pd.read_csv('C:/Users/MAHE/Documents/Mahesh/model 6/model 6(2)/256.csv')
X=df.iloc[:,0:256].values
y=df.iloc[:,-1].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=5)
neigh.fit(X_train, y_train)
y_pred=neigh.predict(X_test)

Acc = accuracy_score(y_test, y_pred),
precision = precision_score(y_test, y_pred, average='weighted'),
recall = recall_score(y_test, y_pred, average='weighted'),
f1 = f1_score(y_test, y_pred, average='weighted'),
print('Accuracy',Acc,'Precision',precision,'Recall',recall,'F1',f1)

Accuracy (0.7170212765957447,) Precision (0.6652075965207677,) Recall (0.7170212765957447,) F1 (0.6716221653615637,)


In [31]:
from lazypredict.Supervised import LazyClassifier
clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train, X_test, y_train, y_test)
print(models)

 97%|███████████████████████████████████████████████████████████████████████████████▎  | 30/31 [01:21<00:02,  2.31s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003924 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26480
[LightGBM] [Info] Number of data points in the train set: 3757, number of used features: 131
[LightGBM] [Info] Start training from score -0.308390
[LightGBM] [Info] Start training from score -1.846182
[LightGBM] [Info] Start training from score -2.229961


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [01:22<00:00,  2.67s/it]

                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
XGBClassifier                      0.91               0.81    None      0.91   
LGBMClassifier                     0.91               0.80    None      0.91   
DecisionTreeClassifier             0.85               0.77    None      0.85   
BaggingClassifier                  0.90               0.76    None      0.89   
RandomForestClassifier             0.88               0.71    None      0.87   
ExtraTreesClassifier               0.87               0.71    None      0.86   
ExtraTreeClassifier                0.76               0.63    None      0.77   
KNeighborsClassifier               0.82               0.61    None      0.80   
NearestCentroid                    0.31               0.49    None      0.30   
BernoulliNB                        0.33               0.49    None      0.34   
AdaBoostClassifier                 0.73 

In [37]:
import os
import numpy as np
import tensorflow as tf
import cv2
from patchify import patchify
import albumentations as A
from sklearn.model_selection import train_test_split

def create_data_generators(dir_path, batch_size=16, image_size=(512, 512), patch_size=224, validation_split=0.1, test_split=0.1):
    image_paths = []
    labels = []
    
    # Collect all image paths and labels
    for root, _, files in os.walk(dir_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
                image_paths.append(os.path.join(root, file))
                labels.append(get_label(file))
            
    
    # First split: Training + Validation vs Test
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        image_paths, labels, test_size=test_split, stratify=labels, random_state=42
    )
    
    # Second split: Training vs Validation
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_val_paths, train_val_labels, test_size=validation_split, stratify=train_val_labels, random_state=42
    )

    augmentation = A.Compose([
        A.HorizontalFlip(p=0.75),
        A.VerticalFlip(p=0.75),
        A.augmentations.geometric.rotate.Rotate(limit=15, p=0.5),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(224, 224), always_apply=False, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    ])

    def generator(paths, labels, is_training):
        total_images = len(paths)
        
        while True:
            # Shuffle the dataset for each epoch
            indices = np.arange(total_images)
            if is_training:
                np.random.shuffle(indices)
            
            for start in range(0, total_images, batch_size):
                end = min(start + batch_size, total_images)
                batch_indices = indices[start:end]
                
                batch_images = []
                batch_labels = []
                
                for i in batch_indices:
                    try:
                        img = cv2.imread(paths[i], cv2.IMREAD_COLOR)
                        img = cv2.resize(img, image_size)  # Ensure consistent size
                        if is_training:
                            augmented = augmentation(image=img)
                            img = augmented['image']
                        img = preprocessing(img)  # Assuming this function exists
                        img = combine(img)
                        
                        # Extract patches
                        patches = patchify(img, (patch_size, patch_size, 3), step=patch_size)
                        patches = patches.reshape(-1, patch_size, patch_size, 3)
                        
                        # Preprocess patches
                        patches = preprocess_patches(patches)
                        
                        batch_images.append(patches)
                        batch_labels.append(labels[i])
                    except Exception as e:
                        print(f"Error processing image {paths[i]}: {str(e)}")
                        continue
                
                if not batch_images:
                    continue  # Skip this batch if all images failed to process
                
                X = np.array(batch_images)
                y = tf.keras.utils.to_categorical(batch_labels, num_classes=3)
                
                # Create a dummy tensor for attention weights
                attention_weights = np.zeros((len(batch_labels), X.shape[1], 1))
                
                yield X, (y, attention_weights)

    train_gen = generator(train_paths, train_labels, is_training=True)
    val_gen = generator(val_paths, val_labels, is_training=False)
    test_gen = generator(test_paths, test_labels, is_training=False)

    return train_gen, val_gen, test_gen

# Usage
train_gen, val_gen, test_gen = create_data_generators(dir_train, validation_split=0.1, test_split=0.1)


In [38]:
history = model.fit(train_gen, epochs=70, steps_per_epoch=316,validation_data=val_gen, validation_steps=40,callbacks=[early_stopping,reduce_lr]) #callbacks=[early_stopping]

Epoch 1/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 1037s 3s/step - accuracy: 0.7142 - loss: 0.8026 - val_accuracy: 0.7417 - val_loss: 0.7814 - learning_rate: 0.0010
Epoch 2/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 989s 3s/step - accuracy: 0.7245 - loss: 0.7925 - val_accuracy: 0.7274 - val_loss: 0.7253 - learning_rate: 0.0010
Epoch 3/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 995s 3s/step - accuracy: 0.7419 - loss: 0.7453 - val_accuracy: 0.7444 - val_loss: 0.7007 - learning_rate: 0.0010
Epoch 4/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 996s 3s/step - accuracy: 0.7401 - loss: 0.7243 - val_accuracy: 0.7258 - val_loss: 0.7837 - learning_rate: 0.0010
Epoch 5/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 1001s 3s/step - accuracy: 0.7226 - loss: 0.7983 - val_accuracy: 0.7428 - val_loss: 0.7405 - learning_rate: 0.0010
Epoch 6/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 922s 3s/step - accuracy: 0.7394 - loss: 0.7517 - val_accuracy: 0.7274 - val_loss: 0.7746 - learning_rate: 0.0010
Epoch 7/70
316/316 ━━━━━━━━━━━━━━━━━━━━ 961s 3s/step - accuracy: 0.7437 - loss: 

In [39]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf

def evaluate_model_on_test_set(model, test_gen, steps, num_classes=3):
    all_labels = []
    all_predictions = []

    # Loop through the test set
    for _ in range(steps):
        X_batch, (y_batch, _) = next(test_gen)
        
        # Predict the output for the batch
        predictions = model.predict(X_batch, verbose=0)
        
        # Get the predicted class (argmax)
        pred_classes = np.argmax(predictions, axis=1)
        
        # Get the true class (argmax)
        true_classes = np.argmax(y_batch, axis=1)
        
        # Append to the lists
        all_labels.extend(true_classes)
        all_predictions.extend(pred_classes)
    
    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    # Calculate accuracy, precision, recall, and F1 score
    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average='weighted')
    recall = recall_score(all_labels, all_predictions, average='weighted')
    f1 = f1_score(all_labels, all_predictions, average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return accuracy, precision, recall, f1

# Assuming you have a test generator and your trained model
# Example usage:
# steps is the number of batches to evaluate. Adjust it according to the size of your test set.
test_steps = 40  # Calculate as len(test_dataset) // batch_size
accuracy, precision, recall, f1 = evaluate_model_on_test_set(model, test_gen, steps=test_steps)


Accuracy: 0.7175
Precision: 0.5147
Recall: 0.7175
F1 Score: 0.5994
